In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import flopy

In [ ]:
outdir = os.path.join("output")

# OBS DATA

In [ ]:
obsdata = pd.read_csv(os.path.join("data", "obs_chem.csv"))


In [ ]:
# for obsid, obs_df in obsdata.groupby('obsid'):
#     vars_ = obs_df['var'].unique()
#     n_vars = len(vars_)

#     fig, axes = plt.subplots(
#         n_vars, 1,
#         figsize=(6.3, 3 * n_vars),
#         sharex=True
#     )
#     if n_vars == 1:                # ensure axes is iterable when only one subplot
#         axes = [axes]
#     for ax, var in zip(axes, vars_):
#         vdf = obs_df[obs_df['var'] == var]
#         ax.scatter(vdf['days'], vdf['value'], marker='o')
#         ax.set_ylabel(f"{var} ({vdf['units'].iloc[0]})")
#         ax.set_title(f"{obsid} – {var}")
#         ax.grid(True)


#     axes[-1].set_xlabel("Days")
#     fig.suptitle(f"Time Series for {obsid}", fontsize=14, y=1.02)
#     fig.tight_layout()
#     # fig.savefig(f"{obsid}.png", dpi=300)    # uncomment to save

#     plt.show()

In [ ]:
ws = os.path.join("model")
sim = flopy.mf6.MFSimulation.load(sim_ws = ws,sim_name = 'gwf', version='mf6', exe_name='mf6' )

gwf = sim.get_model("gwf")

# EXPORT VTK

In [ ]:
obs = flopy.utils.observationfile.Mf6Obs(os.path.join(ws,'obs_Cl.csv'),isBinary=False)
obs.get_nobs()

model_name = 'dizon36'
gwf.dis.idomain.export(os.path.join(outdir, f"{model_name}.vtk"), fmt="vtk")
gwf.wel[0].export(os.path.join(outdir, f"{model_name}.vtk"), fmt="vtk")
gwf.wel[1].export(os.path.join(outdir, f"{model_name}.vtk"), fmt="vtk")
gwf.npf.k.export(os.path.join(outdir, f"{model_name}.vtk"), fmt="vtk")
gwf.chd.export(os.path.join(outdir, f"{model_name}.vtk"), fmt="vtk")

# HEADS

In [ ]:
# Get the time discretization information
tdis = gwf.simulation.tdis
perioddata = tdis.perioddata.get_data()
nlay = gwf.dis.nlay.get_data()


# Calculate cumulative end times from each stress period
cumulative_time = 0
end_of_stress_period_times = []
for perlen in perioddata["perlen"]:
    cumulative_time += perlen
    end_of_stress_period_times.append(cumulative_time)

# plot heads in mapview for each layer
h = gwf.output.head().get_alldata()

# plot the head results mapview
mapview_output_dir = os.path.join(ws, 'output', 'mapview')
if not os.path.exists(mapview_output_dir):
    os.makedirs(mapview_output_dir)


mapview_counter = 0
for t in range(38, 39):
    h_t = h[t, :, :, :]
    masked_h = np.ma.masked_where((h_t < -500) | (h_t > 500), h_t)
    vmin = masked_h.min()
    vmax = masked_h.max()
    vmin, vmax = np.nanmin(masked_h), np.nanmax(masked_h)  # Set colorbar limits

    contour_intervals = np.linspace(vmin, vmax, 20)  # Contour levels

    #contour_intervals = np.arange(vmin, vmax, (vmax-vmin)/10)
    
    # Create a figure with 4 rows and 3 columns
    fig, axes = plt.subplots(4, 3, figsize=(16, 16), constrained_layout=True)
    fig.subplots_adjust(right=0.85)  # Leave space for the colorbar
    
    # Loop over all layers
    for i, ax in enumerate(axes.flat):
        if i < nlay:
            # Plot for layer `i`
            ax.set_title(f"Model Layer {i + 1}")
            modelmap = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=i)
    
            # Plot array, boundary conditions, and contours
            pa = modelmap.plot_array(masked_h[i, :, :], vmin=vmin, vmax=vmax)
            modelmap.plot_bc("WEL", color='black')
            modelmap.plot_bc("CHD", color='blue')
            modelmap.plot_grid(lw=0.5, color="0.5")
            contours = modelmap.contour_array(
                masked_h[i, :, :],
                levels=contour_intervals,
                colors="black",
            )
            ax.clabel(contours, fmt="%2.1f")
        else:
            # Hide axes for extra subplots
            ax.axis("off")
    
    # Add a single colorbar for the entire figure
    cbar_ax = fig.add_axes([1.01, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
    fig.colorbar(pa, cax=cbar_ax, label="Head (mamsl)")
    plt.suptitle('Simulated Head at Stress Period: ' + str(t+1) + ' time in days: ' + str(end_of_stress_period_times[t]))
    # Use tight_layout for clean subplot spacing
    #fig.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to leave space for the colorbar
    # fig.savefig(os.path.join('000' + str(mapview_counter) + '_SimulatedHeads_mamsl_nper_' + str(t+1) + '_time_days_' + str(end_of_stress_period_times[t]) + '.png'), dpi=400, bbox_inches='tight')
    #plt.close(fig)
    mapview_counter+=1
    plt.show()

# DIS MAPS AND XSECTIONS

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 4), constrained_layout=True)
modelmap = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line={"row": 9},geographic_coords=True)
# pa = modelmap.plot_array(h, vmin=vmin, vmax=vmax)
quadmesh = modelmap.plot_bc("WEL")
quadmesh2 = modelmap.plot_bc("chd")
linecollection = modelmap.plot_grid(lw=0.5, color="0.5")


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 4), constrained_layout=True)
modelmap = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=1)
# pa = modelmap.plot_array(h, vmin=vmin, vmax=vmax)
quadmesh = modelmap.plot_bc("WEL")
linecollection = modelmap.plot_grid(lw=0.5, color="0.5")


# OBS VS SIM

# Cl non reactive transport

In [ ]:
ws = os.path.join("model")
sim = flopy.mf6.MFSimulation.load(sim_ws = ws,sim_name = 'gwf', version='mf6', exe_name='mf6' )
gwf = sim.get_model("gwf")

obsdata = pd.read_csv(os.path.join("data", "obs_chem.csv"))
obsdata['obsid'] = obsdata['obsid'].str.lower()
cldata = obsdata[obsdata['var'] == 'Cl']

cl = pd.read_csv(os.path.join(ws, "obs_Cl.csv"))
cl = cl.melt(var_name='obsid', value_name=f'cl', id_vars=['time'])
cl['obsid'] = cl['obsid'].str.lower()

In [ ]:

cl['wp'] = cl['obsid'].str.extract(r'(wp\d+)', expand=False)
cl['f']  = cl['obsid'].str.extract(r'(f\d+)',  expand=False)

# Drop rows you don’t want (already removed f5 above)
cl = cl[~cl['f'].eq('f5')]
cl = cl[~cl['wp'].eq('wp4')]

wps = sorted(cl['wp'].unique(), key=lambda s: int(s[2:]))   # wp1, wp2, …
fs  = sorted(cl['f'].unique(),  key=lambda s: int(s[1:]))   # f1,  f2,  …

nrow, ncol = len(fs), len(wps)

fig, axes = plt.subplots(
    nrow, ncol,
    figsize=(3.2 * ncol, 2.2 * nrow),
    sharex='col', sharey='row'
)
# axes is a 2‑D array even if any dimension is 1
axes = np.atleast_2d(axes)

# ------------------------------------------------------------------
# 2)  Loop over the grid
# ------------------------------------------------------------------
for r, f in enumerate(fs):
    for c, wp in enumerate(wps):
        ax = axes[r, c]

        mask  = (cl['wp'] == wp) & (cl['f'] == f)
        lines = cl.loc[mask]
        if not lines.empty:
            ax.plot(lines['time'], lines['cl'], lw=1)

            data_pts = cldata[
                (cldata['obsid'].str.contains(wp, case=False)) &
                (cldata['obsid'].str.contains(f,  case=False))
            ]
            if not data_pts.empty:
                ax.scatter(data_pts['days'], data_pts['value'], s=10)

        # Formatting
        if r == 0:                       # top row → column titles
            ax.set_title(wp.upper())
        if c == 0:                       # first col → y‑label per row
            ax.set_ylabel(f.upper() + "\nCl (mol/L)")
        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.6)

# Shared labels
for ax in axes[-1, :]:
    ax.set_xlabel("Time")

fig.tight_layout()
plt.show()
